# Embeddings in NumPy (optional)
**LLMs, Animated · Lesson P3.4**

Work with a toy embedding table: show that a one-hot multiply is a row lookup, measure cosine similarity, find nearest neighbours, try the famous analogy (and see why the "exclude the inputs" rule matters), and compute tied output logits.

Each exercise has a `# TODO` cell and an `assert` check cell. Solutions are at the bottom. Only `numpy` is needed.

*Note:* these vectors are **hand-built toys** with readable features, so you can see what's going on. Real embeddings are learned, have hundreds or thousands of dimensions, and their directions rarely have tidy names.

In [ ]:
import numpy as np
np.random.seed(0)

# features:          animal fruit vehicle royal female  size
toy = {
    "cat":      [0.9, 0.0, 0.0, 0.0, 0.5, 0.2],
    "dog":      [0.9, 0.0, 0.0, 0.0, 0.5, 0.3],
    "horse":    [0.8, 0.0, 0.1, 0.0, 0.5, 0.7],
    "apple":    [0.0, 0.9, 0.0, 0.0, 0.5, 0.1],
    "banana":   [0.0, 0.9, 0.0, 0.0, 0.5, 0.2],
    "car":      [0.0, 0.0, 0.9, 0.0, 0.5, 0.6],
    "bus":      [0.0, 0.0, 0.9, 0.0, 0.5, 0.9],
    "man":      [0.3, 0.0, 0.0, 0.1, 0.0, 0.5],
    "woman":    [0.3, 0.0, 0.0, 0.1, 1.0, 0.5],
    "king":     [0.3, 0.0, 0.0, 0.9, 0.0, 0.5],
    "queen":    [0.3, 0.0, 0.0, 0.9, 1.0, 0.5],
}
vocab = list(toy)
idx = {w: i for i, w in enumerate(vocab)}
E = np.array([toy[w] for w in vocab]) + np.random.normal(0, 0.03, (len(vocab), 6))  # V x d
V, d = E.shape
print("E has shape", E.shape)

## 1 · A one-hot multiply is a row lookup
Write `one_hot(i, V)` returning a length-`V` vector with a 1 at position `i`, and `embed(ids, E)` that returns the embedding vectors for a list of token IDs **by indexing** (no matrix multiply).

In [ ]:
def one_hot(i, V):
    # TODO
    raise NotImplementedError("Exercise 1a")

def embed(ids, E):
    # TODO: return an array of shape (len(ids), d)
    raise NotImplementedError("Exercise 1b")

**Check 1**

In [ ]:
ids = [idx["cat"], idx["dog"], idx["bus"]]
X = np.stack([one_hot(i, V) for i in ids])          # (3, V) one-hot rows
assert X.shape == (3, V) and X.sum() == 3
assert np.allclose(X @ E, embed(ids, E)), "one-hot times E should equal the row lookup"
print("✓ exercise 1: one-hot @ E picks out rows")

## 2 · Cosine similarity
$$\cos(a,b) = \frac{a\cdot b}{\lVert a\rVert\,\lVert b\rVert}$$

In [ ]:
def cosine(a, b):
    # TODO
    raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
assert abs(cosine(np.array([1, 2, 2]), np.array([2, 1, 2])) - 8 / 9) < 1e-9   # the lesson's worked example
assert abs(cosine(np.array([3, 4]), np.array([4, 3])) - 0.96) < 1e-9
assert abs(cosine(np.array([1, 0]), np.array([0, 5]))) < 1e-9
assert abs(cosine(np.array([1, 1]), np.array([-2, -2])) + 1) < 1e-9
v = lambda w: E[idx[w]]
print("cat·dog", round(cosine(v("cat"), v("dog")), 3), " cat·car", round(cosine(v("cat"), v("car")), 3))
assert cosine(v("cat"), v("dog")) > cosine(v("cat"), v("car"))
print("✓ exercise 2")

## 3 · Nearest neighbours
Write `nearest(vec, k, exclude=())` returning the `k` vocabulary words whose vectors have the highest cosine similarity with `vec`, best first, skipping any word in `exclude`.

In [ ]:
def nearest(vec, k=3, exclude=()):
    # TODO: score every word with cosine, sort descending, skip excluded words
    raise NotImplementedError("Exercise 3")

**Check 3**

In [ ]:
assert nearest(v("cat"), 1, exclude=["cat"]) == ["dog"]
assert set(nearest(v("apple"), 1, exclude=["apple"])) == {"banana"}
assert nearest(v("bus"), 1) == ["bus"], "without exclusion, a word is its own nearest neighbour"
print("✓ exercise 3:", nearest(v("horse"), 3, exclude=["horse"]))

## 4 · The analogy, with and without the rule
Write `analogy(a, b, c, exclude_inputs=True)` that answers "*a* is to *b* as *c* is to ?" by finding the word nearest to $b - a + c$. With `exclude_inputs=True`, the three input words may not be the answer (that's the standard rule).

In [ ]:
def analogy(a, b, c, exclude_inputs=True):
    # TODO: target = v(b) - v(a) + v(c); return the single nearest word
    raise NotImplementedError("Exercise 4")

**Check 4**

In [ ]:
assert analogy("man", "king", "woman") == "queen"
print("with the rule:   man : king :: woman :", analogy("man", "king", "woman"))
print("without the rule:", analogy("man", "king", "woman", exclude_inputs=False),
      "| top 3:", nearest(v("king") - v("man") + v("woman"), 3))
print("✓ exercise 4")

Here the toy vectors are built so the analogy works perfectly. In real word2vec embeddings, Nissim et al. (2020) found that without the exclusion rule the answer is often one of the input words, such as "king" itself. Always check which rule a reported analogy result used.

## 5 · Tied output logits
With **weight tying** the unembedding is $E^\top$, so the logits for a final hidden vector $h$ are $h E^\top$ (one dot product per token). Write `next_token_probs(h, E)` returning the softmax of those logits. Use the stable softmax from P3.2 (subtract the max).

In [ ]:
def next_token_probs(h, E):
    # TODO: logits = h @ E.T, then a numerically stable softmax
    raise NotImplementedError("Exercise 5")

**Check 5**

In [ ]:
p = next_token_probs(v("dog") * 5, E)
assert p.shape == (V,) and abs(p.sum() - 1) < 1e-9 and (p > 0).all()
print("top 3:", [(vocab[i], round(float(p[i]), 3)) for i in np.argsort(-p)[:3]])
assert vocab[int(np.argmax(p))] in ("dog", "cat", "horse")
assert np.isfinite(next_token_probs(v("dog") * 1000, E)).all(), "use the subtract-the-max trick"
print("✓ exercise 5")

Notice that **horse** can beat **dog** even though `h` was built from dog's vector. Logits are dot products, not cosines, so a token with a longer vector gets a boost. Real models learn to use vector length as well as direction.

## Experiment
- Add more noise to `E` (change `0.03` to `0.3` in the setup cell and re-run). When does the analogy break?
- Scale `h` in exercise 5 by 1, 5 and 50. What happens to the distribution? (Compare with temperature in P3.2.)
- Which word is nearest to `v("king") - v("man")`? What does that direction "mean" in this toy?

In [ ]:
# your experiments here


## Recap
- An embedding layer is a $V \times d$ table; one-hot × table = row lookup.
- Cosine similarity compares directions and ignores length.
- Analogy arithmetic depends on the evaluation rule (exclude the inputs) and on the data.
- With weight tying, output logits are dot products with the same token vectors.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def one_hot(i, V):
    x = np.zeros(V)
    x[i] = 1.0
    return x

def embed(ids, E):
    return E[np.asarray(ids)]

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def cosine(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def nearest(vec, k=3, exclude=()):
    scores = [(cosine(vec, E[i]), w) for i, w in enumerate(vocab) if w not in exclude]
    scores.sort(reverse=True)
    return [w for _, w in scores[:k]]

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def analogy(a, b, c, exclude_inputs=True):
    target = v(b) - v(a) + v(c)
    return nearest(target, 1, exclude=(a, b, c) if exclude_inputs else ())[0]

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def next_token_probs(h, E):
    logits = h @ E.T
    z = logits - logits.max()
    e = np.exp(z)
    return e / e.sum()